# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [33]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
OPENAI_API_KEY="xx"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
TAVILY_API_KEY="xx"

In [4]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

In [8]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.create_collection(
   name="UdaplayGames",
   embedding_function=embedding_fn
)

### Add documents

In [9]:
# Make sure you have a directory "project/starter/games"
data_dir = "starter/games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

## TESTS FOR EACH MODULE

### Setup Validation Test


In [34]:
import os
import sqlite3
from dotenv import load_dotenv
from chromadb.utils import embedding_functions
import chromadb

load_dotenv()

print("UdaPlay — Setup Validation")
print("-" * 35)

checks_passed = 0
checks_total = 0

def check(name, passed, details=""):
    global checks_passed, checks_total
    checks_total += 1

    if passed:
        checks_passed += 1
        print(f"✓ PASS: {name}" + (f" — {details}" if details else ""))
    else:
        print(f"✗ FAIL: {name}" + (f" — {details}" if details else ""))

# 1. Required packages
for package in ["chromadb", "dotenv"]:
    try:
        __import__(package)
        check(f"Import {package}", True)
    except Exception as e:
        check(f"Import {package}", False, str(e))

# 2. Embedding functions
try:
    _ = embedding_functions.OpenAIEmbeddingFunction
    check("ChromaDB embedding functions", True)
except Exception as e:
    check("ChromaDB embedding functions", False, str(e))

# 3. SQLite
try:
    with sqlite3.connect(":memory:") as connection:
        result = connection.execute("SELECT 1").fetchone()
        assert result == (1,)
    check("SQLite database", True)
except Exception as e:
    check("SQLite database", False, str(e))

# 4. Environment configuration
check(
    ".env file",
    os.path.isfile(".env"),
    "Found" if os.path.isfile(".env") else "Not found in current directory"
)

# 5. API keys (values are never printed)
for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    check(
        f"{key} configured",
        bool(os.getenv(key)),
        "Available" if os.getenv(key) else "Missing or not loaded"
    )

print("-" * 35)
print(f"Result: {checks_passed}/{checks_total} checks passed")

if checks_passed == checks_total:
    print("✓ PASS: Setup is ready!")
else:
    print(f"✗ FAIL: {checks_total - checks_passed} check(s) need attention.")


UdaPlay — Setup Validation
-----------------------------------
✓ PASS: Import chromadb
✓ PASS: Import dotenv
✓ PASS: ChromaDB embedding functions
✓ PASS: SQLite database
✓ PASS: .env file — Found
✓ PASS: OPENAI_API_KEY configured — Available
✓ PASS: TAVILY_API_KEY configured — Available
-----------------------------------
Result: 7/7 checks passed
✓ PASS: Setup is ready!


### Embedding Function & Collection Setup Test

In [26]:

import os
from dotenv import load_dotenv
from chromadb.utils import embedding_functions

load_dotenv()

# 1. Check the OpenAI API key is available
api_key = os.getenv("OPENAI_API_KEY")
assert api_key, "OPENAI_API_KEY is missing. Check your .env file."

# 2. Create the embedding function
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name="text-embedding-3-small"
)
print("OpenAI embedding function created")

# 3. Create or load the collection
collection = chroma_client.get_or_create_collection(
    name="UdaplayGames",
    embedding_function=embedding_fn
)
print("ChromaDB collection ready:", collection.name)

# 4. Test embedding, storage, and retrieval
try:
    collection.upsert(
        ids=["udaplay_setup_test"],
        documents=["UdaPlay setup test: an adventure video game"]
    )

    result = collection.query(
        query_texts=["adventure video game"],
        n_results=1
    )

    assert result["documents"][0], "No document was retrieved"

    print("✓ PASS: Embedding generation works")
    print("✓ PASS: Document storage works")
    print("✓ PASS: Similarity search works")
    print("✓ Retrieved:", result["documents"][0][0])

finally:
    # Remove only the temporary test record
    collection.delete(ids=["udaplay_setup_test"])


OpenAI embedding function created
ChromaDB collection ready: UdaplayGames
✓ PASS: Embedding generation works
✓ PASS: Document storage works
✓ PASS: Similarity search works
✓ Retrieved: UdaPlay setup test: an adventure video game


### Game Data Indexing & Validation Test

In [31]:
import os
import json

data_dir = "starter/games"

# 1. Check that the games directory exists
assert os.path.isdir(data_dir), (
    f"Games directory not found: {data_dir}. "
    "Check your notebook's working directory."
)

json_files = sorted(
    f for f in os.listdir(data_dir) if f.lower().endswith(".json")
)

assert json_files, f"No JSON files found in {data_dir}"

print(f"Found {len(json_files)} JSON file(s)")

indexed_count = 0

# 2. Read and index game data
for file_name in json_files:
    file_path = os.path.join(data_dir, file_name)

    with open(file_path, "r", encoding="utf-8") as f:
        games = json.load(f)

    # Support both a single game object and a list of games
    if isinstance(games, dict):
        games = [games]

    assert isinstance(games, list), (
        f"Unexpected JSON structure in {file_name}"
    )

    for i, game in enumerate(games):
        required_fields = [
            "Platform", "Name", "YearOfRelease", "Description"
        ]
        missing = [field for field in required_fields if field not in game]

        assert not missing, (
            f"x Missing fields in {file_name}: {missing}"
        )

        content = (
            f"[{game['Platform']}] {game['Name']} "
            f"({game['YearOfRelease']}) - {game['Description']}"
        )

        # Unique ID for each game, even if a file contains multiple games
        base_id = os.path.splitext(file_name)[0]
        doc_id = base_id if len(games) == 1 else f"{base_id}_{i}"

        # Chroma metadata values must be scalar types
        metadata = {
            key: value if isinstance(value, (str, int, float, bool))
            else json.dumps(value, ensure_ascii=False)
            for key, value in game.items()
            if value is not None
        }

        collection.upsert(
            ids=[doc_id],
            documents=[content],
            metadatas=[metadata]
        )

        indexed_count += 1

print(f"\n ✓ PASS: Indexed {indexed_count} game(s)")
print(f"Total documents in collection: {collection.count()}")

# 3. Test retrieval
result = collection.query(
    query_texts=["video game"],
    n_results=1
)

assert result["documents"] and result["documents"][0], (
    "Retrieval test failed: no documents returned"
)

print("✓ PASS: Vector database retrieval works")
print("Example retrieved document:")
print(result["documents"][0][0])


Found 15 JSON file(s)

 ✓ PASS: Indexed 15 game(s)
Total documents in collection: 15
✓ PASS: Vector database retrieval works
Example retrieved document:
[PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
